In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report




In [37]:
df=pd.read_csv(r"C:\Users\lakshmilokeswari\Downloads\disaster_messages_without_other.csv")

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12154 entries, 0 to 12153
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   message   12154 non-null  object
 1   category  12154 non-null  object
dtypes: object(2)
memory usage: 190.0+ KB


In [41]:
df.dropna()

,message,category
0,UN reports Leogane 80-90 destroyed. Only Hospi...,medical_help
1,"Please, we need tents and water. We are in Sil...",flood
2,"There's nothing to eat and water, we starving ...",flood
3,"I am in Thomassin number 32, in the area named...",flood
4,"Let's do it together, need food in Delma 75, i...",food_help
...,...,...
12149,Following the severe floods which occurred ove...,flood
12150,"BANGKOK, 24 January 2012 (NNT) - Prime Ministe...",flood
12151,"Cadmium, a metallic element widely used in bat...",infrastructure_damage
12152,"Hpakant, an area rich with coveted jade stones...",fire


In [44]:
#4. TEXT CLEANING FUNCTION
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean(doc):
    # Remove numbers and special characters (keep only alphabets)
    doc = re.sub(r'[^a-zA-Z\s]', ' ', doc)

    # Convert to lowercase
    doc = doc.lower()

    # Tokenization
    tokens = nltk.word_tokenize(doc)

    # Stopword removal
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Remove extra spaces and join
    return " ".join(tokens)


In [45]:
df['clean_text']=df['message'].apply(clean)
df.head()

,message,category,clean_text
0,UN reports Leogane 80-90 destroyed. Only Hospi...,medical_help,un report leogane destroyed hospital st croix ...
1,"Please, we need tents and water. We are in Sil...",flood,please need tent water silo thank
2,"There's nothing to eat and water, we starving ...",flood,nothing eat water starving thirsty
3,"I am in Thomassin number 32, in the area named...",flood,thomassin number area named pyron would like w...
4,"Let's do it together, need food in Delma 75, i...",food_help,let together need food delma didine area


In [46]:


X = df["message"]
y = df["category"]

In [47]:
#  TRAIN TEST SPLIT
# =========================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(9723,) (2431,)
(9723,) (2431,)


In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b',
    min_df=5,
    max_df=0.75,
    ngram_range=(1,2),
    norm="l2"
)

In [50]:
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

In [51]:
tfidf_df = pd.DataFrame(
    X_train_vec.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

,abandoned,ability,able,able help,abroad,absence,absorb,abundant,accelerated,access,...,yunnan,yunnan province,zambia,zero,zhejiang,zhouqu,zimbabwe,zinc,zone,zones
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [52]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

param_grid = {
    "alpha": [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]
}

grid_mnb = GridSearchCV(
    MultinomialNB(),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_mnb.fit(X_train_vec, y_train)

best_mnb = grid_mnb.best_estimator_

print("Best alpha:", grid_mnb.best_params_)

Best alpha: {'alpha': 0.1}


In [53]:
y_pred_mnb = best_mnb.predict(X_test_vec)

print(
    "Tuned MultinomialNB Accuracy:",
    accuracy_score(y_test, y_pred_mnb)
)

Tuned MultinomialNB Accuracy: 0.7832167832167832


In [55]:
print(
    classification_report(
        y_test,
        y_pred_mnb
    )
)


                       precision    recall  f1-score   support

           earthquake       0.90      0.84      0.87       389
                 fire       0.89      0.30      0.44        54
                flood       0.74      0.92      0.82       980
            food_help       0.78      0.73      0.75       341
infrastructure_damage       0.78      0.50      0.61       201
         medical_help       0.80      0.49      0.61       120
          rescue_help       0.79      0.72      0.75       346

             accuracy                           0.78      2431
            macro avg       0.81      0.64      0.69      2431
         weighted avg       0.79      0.78      0.77      2431



In [56]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "metric": ["cosine"],
    "weights": ["uniform", "distance"]
}

In [57]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

knn = KNeighborsClassifier()

grid_knn = GridSearchCV(
    knn,
    param_grid_knn,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_knn.fit(X_train_vec, y_train)


GridSearchCV(cv=5, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'metric': ['cosine'], 'n_neighbors': [3, 5, 7, 9, 11],
                         'weights': ['uniform', 'distance']},
             scoring='accuracy')

In [58]:
best_knn = grid_knn.best_estimator_

print("Best KNN params:", grid_knn.best_params_)

Best KNN params: {'metric': 'cosine', 'n_neighbors': 11, 'weights': 'distance'}


In [59]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_knn = best_knn.predict(X_test_vec)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

KNN Accuracy: 0.7387906211435623
                       precision    recall  f1-score   support

           earthquake       0.77      0.85      0.81       389
                 fire       0.94      0.30      0.45        54
                flood       0.78      0.83      0.80       980
            food_help       0.69      0.72      0.70       341
infrastructure_damage       0.64      0.48      0.55       201
         medical_help       0.82      0.48      0.61       120
          rescue_help       0.64      0.69      0.67       346

             accuracy                           0.74      2431
            macro avg       0.75      0.62      0.66      2431
         weighted avg       0.74      0.74      0.73      2431



In [60]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier()

param_grid_knn = {
    "n_neighbors": list(range(1, 50))
}

grid_knn = GridSearchCV(
    knn,
    param_grid_knn,
    cv=5,
    scoring="accuracy"
)

# CHANGED: X_train_tfidf -> X_train_vec
grid_knn.fit(X_train_vec, y_train)

print("Best K:", grid_knn.best_params_)
print("Best CV Accuracy (KNN):", grid_knn.best_score_)

best_knn = grid_knn.best_estimator_

# CHANGED: X_test_tfidf -> X_test_vec
y_pred_knn = best_knn.predict(X_test_vec)

print("KNN Test Accuracy:", accuracy_score(y_test, y_pred_knn))

Best K: {'n_neighbors': 43}
Best CV Accuracy (KNN): 0.7607747521872057
KNN Test Accuracy: 0.7675853558206499


In [61]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Decision Tree model
dt = DecisionTreeClassifier(random_state=42)

# Hyperparameter grid (UNCHANGED)
param_grid_dt = {
    "max_depth": list(range(1, 50))
}

# GridSearchCV
grid_dt = GridSearchCV(
    dt,
    param_grid_dt,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Fit model
grid_dt.fit(X_train_vec, y_train)

# Best parameters & CV score
print("Best Depth:", grid_dt.best_params_)
print("Best CV Accuracy (DT):", grid_dt.best_score_)

# Best model
best_dt = grid_dt.best_estimator_

# Predictions
y_pred_dt = best_dt.predict(X_test_vec)

# Test Accuracy
print("Decision Tree Test Accuracy:", accuracy_score(y_test, y_pred_dt))

# Classification Report
print("\nDecision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))


Best Depth: {'max_depth': 47}
Best CV Accuracy (DT): 0.9384964613285092
Decision Tree Test Accuracy: 0.9378856437679967

Decision Tree Classification Report:
                       precision    recall  f1-score   support

           earthquake       1.00      0.98      0.99       389
                 fire       0.71      0.44      0.55        54
                flood       0.93      0.95      0.94       980
            food_help       0.96      0.98      0.97       341
infrastructure_damage       0.87      0.89      0.88       201
         medical_help       0.89      0.88      0.89       120
          rescue_help       0.95      0.95      0.95       346

             accuracy                           0.94      2431
            macro avg       0.90      0.87      0.88      2431
         weighted avg       0.94      0.94      0.94      2431



In [63]:
print("\nFINAL MODEL COMPARISON")
print("----------------------")
print("Naive Bayes:", accuracy_score(y_test, y_pred_mnb))
print("KNN:", accuracy_score(y_test, y_pred_knn))
print("Decision Tree:", accuracy_score(y_test, y_pred_dt))


FINAL MODEL COMPARISON
----------------------
Naive Bayes: 0.7832167832167832
KNN: 0.7675853558206499
Decision Tree: 0.9378856437679967
